# RAG-Powered Drug Leaflet Assistant — Pipeline Notebook

This notebook builds the full RAG pipeline: collects source documents, chunks them, generates embeddings, builds and evaluates retrieval, and persists a vector store for the FastAPI backend to load.

**Domain:** drug package insert leaflets (dosage, warnings, interactions, side effects), sourced from the FDA DailyMed API.

**Track:** Core (text-only RAG).


## Phase 0 — Environment Setup

Installs the packages needed for this notebook: PDF text extraction, embeddings, and the vector store.


In [1]:
!pip install -q pypdf chromadb sentence-transformers ollama requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

## Phase 1 — Domain & Data Collection

**Domain:** 30 real drug package insert leaflets spanning pain relief, antibiotics, cardiovascular, diabetes, allergy/respiratory, mental health, and GI medications — collected programmatically from the [FDA DailyMed](https://dailymed.nlm.nih.gov) API.


In [2]:
import csv
import re
import time
from pathlib import Path

import requests

BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"
OUTPUT_DIR = Path("data/raw")
LOG_PATH = OUTPUT_DIR / "collection_log.csv"
REQUEST_DELAY_SECONDS = 1.0  # be polite to a free public API

DRUG_LIST = [
    # Pain relief / NSAIDs
    "Ibuprofen", "Acetaminophen", "Aspirin", "Naproxen", "Diclofenac",
    # Antibiotics
    "Amoxicillin", "Azithromycin", "Ciprofloxacin", "Doxycycline", "Metronidazole",
    # Blood pressure / cardiovascular
    "Lisinopril", "Amlodipine", "Losartan", "Metoprolol", "Atorvastatin",
    # Diabetes
    "Metformin", "Glipizide", "Insulin glargine", "Sitagliptin",
    # Allergy / respiratory
    "Cetirizine", "Loratadine", "Albuterol", "Montelukast",
    # Mental health / neuro
    "Sertraline", "Fluoxetine", "Alprazolam", "Gabapentin",
    # GI / other common
    "Omeprazole", "Loperamide", "Levothyroxine",
]


def sanitize_filename(name: str) -> str:
    """Turn a drug name into a safe filename."""
    name = name.strip().lower().replace(" ", "_")
    return re.sub(r"[^a-z0-9_\-]", "", name)


def search_spl(drug_name: str) -> dict | None:
    """
    Query the DailyMed SPL search endpoint for a drug name.
    Returns the first matching SPL record (dict with 'setid', 'title', ...)
    or None if nothing was found.
    """
    resp = requests.get(
        f"{BASE}/spls.json",
        params={"drug_name": drug_name, "pagesize": 5, "page": 1},
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    results = data.get("data", [])
    return results[0] if results else None


def download_pdf(setid: str, out_path: Path) -> bool:
    """Download the label PDF for a given SET ID to out_path. Returns success."""
    url = "https://dailymed.nlm.nih.gov/dailymed/downloadpdffile.cfm"
    resp = requests.get(url, params={"setId": setid}, timeout=30)
    if resp.status_code != 200 or not resp.content:
        return False
    out_path.write_bytes(resp.content)
    return True


def collect_all_leaflets() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    log_rows = []
    successes = 0

    for i, drug in enumerate(DRUG_LIST, start=1):
        print(f"[{i}/{len(DRUG_LIST)}] Searching for '{drug}'...")
        status = "failed_search"
        setid = ""
        title = ""
        filename = ""

        try:
            spl = search_spl(drug)
            if spl is None:
                print(f"  -> no SPL found for '{drug}', skipping.")
            else:
                setid = spl.get("setid", "")
                title = spl.get("title", drug)
                filename = f"{sanitize_filename(drug)}.pdf"
                out_path = OUTPUT_DIR / filename

                time.sleep(REQUEST_DELAY_SECONDS)
                ok = download_pdf(setid, out_path)

                if ok:
                    status = "downloaded"
                    successes += 1
                    print(f"  -> saved {out_path} (setid={setid})")
                else:
                    status = "failed_download"
                    print(f"  -> found setid={setid} but PDF download failed.")

        except requests.RequestException as exc:
            status = f"error: {exc}"
            print(f"  -> request error for '{drug}': {exc}")

        log_rows.append({
            "drug_name": drug, "setid": setid, "matched_title": title,
            "filename": filename, "status": status,
        })
        time.sleep(REQUEST_DELAY_SECONDS)

    with open(LOG_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["drug_name", "setid", "matched_title", "filename", "status"])
        writer.writeheader()
        writer.writerows(log_rows)

    print(f"\nDone: {successes}/{len(DRUG_LIST)} leaflets downloaded.")
    print(f"Log written to {LOG_PATH}")


collect_all_leaflets()


[1/30] Searching for 'Ibuprofen'...
  -> saved data/raw/ibuprofen.pdf (setid=90659e83-d989-46a5-9680-3a4f61612511)
[2/30] Searching for 'Acetaminophen'...
  -> saved data/raw/acetaminophen.pdf (setid=077fac1a-0ec5-40d9-8861-a281b48f650a)
[3/30] Searching for 'Aspirin'...
  -> saved data/raw/aspirin.pdf (setid=7ce4807e-7bd5-41f1-a9c7-1f3af6acdc09)
[4/30] Searching for 'Naproxen'...
  -> saved data/raw/naproxen.pdf (setid=f6f4a670-235e-4235-84a8-a74dc8ca5586)
[5/30] Searching for 'Diclofenac'...
  -> saved data/raw/diclofenac.pdf (setid=30c3c3c5-b4f3-4321-ae25-b2c588ad0551)
[6/30] Searching for 'Amoxicillin'...
  -> saved data/raw/amoxicillin.pdf (setid=4adc198e-39d8-4b3e-bfa4-278aaf4064ae)
[7/30] Searching for 'Azithromycin'...
  -> saved data/raw/azithromycin.pdf (setid=814abe38-da4a-48af-bf4b-2ce04a5aabd1)
[8/30] Searching for 'Ciprofloxacin'...
  -> saved data/raw/ciprofloxacin.pdf (setid=c47250c2-bece-46b5-8b3b-b7c97d9005d8)
[9/30] Searching for 'Doxycycline'...
  -> saved data/raw/

### 2.1 Load & Inspect

Verify every collected PDF is text-extractable (not a scanned image needing OCR) before chunking.


In [3]:
from pypdf import PdfReader

for pdf_path in sorted(Path("data/raw").glob("*.pdf")):
    reader = PdfReader(pdf_path)
    text = "".join(page.extract_text() or "" for page in reader.pages)
    print(f"{pdf_path.name}: {len(reader.pages)} pages, {len(text)} chars extracted")
    if len(text) < 200:
        print(f"  WARNING: very little text extracted -- may need OCR")


acetaminophen.pdf: 8 pages, 10087 chars extracted
albuterol.pdf: 28 pages, 46712 chars extracted
alprazolam.pdf: 25 pages, 74703 chars extracted
amlodipine.pdf: 27 pages, 64785 chars extracted
amoxicillin.pdf: 19 pages, 39865 chars extracted
aspirin.pdf: 6 pages, 6039 chars extracted
atorvastatin.pdf: 47 pages, 95497 chars extracted
azithromycin.pdf: 16 pages, 82565 chars extracted
cetirizine.pdf: 5 pages, 3542 chars extracted
ciprofloxacin.pdf: 56 pages, 136155 chars extracted
diclofenac.pdf: 25 pages, 75805 chars extracted
doxycycline.pdf: 15 pages, 34060 chars extracted
fluoxetine.pdf: 72 pages, 234116 chars extracted
gabapentin.pdf: 36 pages, 80343 chars extracted
glipizide.pdf: 11 pages, 26417 chars extracted
ibuprofen.pdf: 6 pages, 5774 chars extracted
insulin_glargine.pdf: 56 pages, 100430 chars extracted
levothyroxine.pdf: 22 pages, 49360 chars extracted
lisinopril.pdf: 23 pages, 55089 chars extracted
loperamide.pdf: 4 pages, 3415 chars extracted
loratadine.pdf: 5 pages, 3510 c

**Result:** 30/30 documents collected via the DailyMed API. All were text-extractable PDFs — no scanned pages or OCR required. Page counts ranged from 4 (loperamide) to 72 (fluoxetine), with a combined ~1.9M characters across the corpus. This range matters for chunking: a fixed chunk-size-with-overlap approach needs to handle very uneven document lengths gracefully.


## Phase 2 — The Notebook: Build & Evaluate the RAG Pipeline

### 2.2 Chunking Strategy

**Approach: section-aware chunking.** DailyMed labels have consistent section headers (Indications, Dosage, Warnings, etc.), so splitting on those headers first — then fixed-size sub-chunking only within sections that are still too long — keeps each chunk topically pure. This matters for a medical domain specifically: a fixed-size-only chunker could slice through the middle of "Dosage and Administration" and merge it with "Warnings," which is a real grounding risk when a wrong section gets cited as the source of a dosage answer.

**Chunk size:** 800 characters, 100 character overlap for any section too long to fit in one chunk — overlap prevents a sentence from being cut in half at a chunk boundary.

**Data-quality finding during development:** the corpus contains **two different label formats**. Prescription drugs use the SPL format (`INDICATIONS AND USAGE`, `DOSAGE AND ADMINISTRATION`, etc.). OTC drugs (aspirin, ibuprofen, acetaminophen, loratadine, cetirizine, loperamide) are legally required to use the FDA's different "Drug Facts" format (`Active ingredient`, `Purpose`, `Uses`, `Warnings`, `Directions`, etc.). The header list below includes both formats so section detection works correctly across the whole corpus.


In [4]:
import re
from pathlib import Path
from pypdf import PdfReader

# Standard section headers used across DailyMed labels -- both the
# prescription (SPL) format and the OTC "Drug Facts" format.
SECTION_HEADERS = [
    # Prescription (SPL) format
    "INDICATIONS AND USAGE",
    "DOSAGE AND ADMINISTRATION",
    "DOSAGE FORMS AND STRENGTHS",
    "CONTRAINDICATIONS",
    "WARNINGS AND PRECAUTIONS",
    "ADVERSE REACTIONS",
    "DRUG INTERACTIONS",
    "USE IN SPECIFIC POPULATIONS",
    "OVERDOSAGE",
    "DESCRIPTION",
    "CLINICAL PHARMACOLOGY",
    "NONCLINICAL TOXICOLOGY",
    "HOW SUPPLIED",
    "STORAGE AND HANDLING",
    "PATIENT COUNSELING INFORMATION",
    # OTC "Drug Facts" format
    "ACTIVE INGREDIENT",
    "PURPOSE",
    "USES",
    "WARNINGS",
    "DIRECTIONS",
    "OTHER INFORMATION",
    "INACTIVE INGREDIENTS",
    "QUESTIONS",
]

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100


def extract_text(pdf_path: Path) -> str:
    reader = PdfReader(pdf_path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def split_into_sections(text: str) -> list[dict]:
    """
    Splits raw label text into (section_name, section_text) pairs by
    locating known headers. Text before the first recognized header
    is kept as an 'UNLABELED' section rather than dropped.
    """
    pattern = "|".join(re.escape(h) for h in SECTION_HEADERS)
    matches = list(re.finditer(rf"^\s*({pattern})\s*$", text, re.MULTILINE | re.IGNORECASE))

    if not matches:
        return [{"section": "UNLABELED", "text": text}]

    sections = []
    if matches[0].start() > 0:
        lead_text = text[: matches[0].start()].strip()
        if lead_text:
            sections.append({"section": "UNLABELED", "text": lead_text})

    for i, match in enumerate(matches):
        section_name = match.group(1).upper()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section_text = text[start:end].strip()
        if section_text:
            sections.append({"section": section_name, "text": section_text})

    return sections


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Fixed-size sub-chunking with overlap, used only when a section is too long."""
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


def build_chunks_for_drug(pdf_path: Path) -> list[dict]:
    """Returns a list of chunk dicts: {drug, section, chunk_id, text}."""
    drug_name = pdf_path.stem
    text = extract_text(pdf_path)
    sections = split_into_sections(text)

    chunks = []
    global_idx = 0  # runs across the whole document -- avoids duplicate
                    # chunk_ids when a section header repeats (e.g. a
                    # boxed warning duplicated as a callout)
    for section in sections:
        sub_chunks = chunk_text(section["text"])
        for sub_chunk in sub_chunks:
            chunks.append({
                "drug": drug_name,
                "section": section["section"],
                "chunk_id": f"{drug_name}_{global_idx}",
                "text": sub_chunk,
            })
            global_idx += 1
    return chunks


all_chunks = []
for pdf_path in sorted(Path("data/raw").glob("*.pdf")):
    drug_chunks = build_chunks_for_drug(pdf_path)
    all_chunks.extend(drug_chunks)
    print(f"{pdf_path.name}: {len(drug_chunks)} chunks across "
          f"{len(set(c['section'] for c in drug_chunks))} sections")

print(f"\nTotal chunks across corpus: {len(all_chunks)}")


acetaminophen.pdf: 20 chunks across 7 sections
albuterol.pdf: 73 chunks across 11 sections
alprazolam.pdf: 113 chunks across 10 sections
amlodipine.pdf: 98 chunks across 9 sections
amoxicillin.pdf: 61 chunks across 10 sections
aspirin.pdf: 15 chunks across 7 sections
atorvastatin.pdf: 147 chunks across 11 sections
azithromycin.pdf: 124 chunks across 10 sections
cetirizine.pdf: 11 chunks across 7 sections
ciprofloxacin.pdf: 206 chunks across 11 sections
diclofenac.pdf: 114 chunks across 10 sections
doxycycline.pdf: 65 chunks across 11 sections
fluoxetine.pdf: 343 chunks across 10 sections
gabapentin.pdf: 122 chunks across 10 sections
glipizide.pdf: 44 chunks across 12 sections
ibuprofen.pdf: 13 chunks across 6 sections
insulin_glargine.pdf: 151 chunks across 10 sections
levothyroxine.pdf: 80 chunks across 12 sections
lisinopril.pdf: 83 chunks across 10 sections
loperamide.pdf: 9 chunks across 6 sections
loratadine.pdf: 10 chunks across 7 sections
losartan.pdf: 116 chunks across 10 secti

### 2.3 Embeddings & Vector Store

**Embedding model: `BAAI/bge-base-en-v1.5`.** Unlike general sentence-similarity models (e.g. MiniLM, mpnet), bge is trained specifically for retrieval — matters here since the corpus is dense medical/technical language (dosages, interactions, contraindications) where precise retrieval directly affects answer correctness. bge recommends prefixing indexed passages with `"passage: "` and queries with `"query: "`, which is applied consistently at both index time (below) and query time (Section 2.4).


In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"
VECTOR_STORE_DIR = "data/vector_store"
COLLECTION_NAME = "drug_leaflets"

model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts_to_embed = [f"passage: {chunk['text']}" for chunk in all_chunks]

print(f"Embedding {len(texts_to_embed)} chunks with {EMBEDDING_MODEL_NAME}...")
embeddings = model.encode(
    texts_to_embed,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Done. Embedding shape: {embeddings.shape}")

# Persist to disk (not in-memory) so the backend can load this exact
# store later without re-embedding anything.
client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)
collection = client.get_or_create_collection(name=COLLECTION_NAME)

collection.add(
    ids=[chunk["chunk_id"] for chunk in all_chunks],
    embeddings=embeddings.tolist(),
    documents=[chunk["text"] for chunk in all_chunks],
    metadatas=[{"drug": chunk["drug"], "section": chunk["section"]} for chunk in all_chunks],
)

print(f"Persisted {collection.count()} chunks to '{VECTOR_STORE_DIR}' "
      f"in collection '{COLLECTION_NAME}'.")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding 2949 chunks with BAAI/bge-base-en-v1.5...


Batches:   0%|          | 0/93 [00:00<?, ?it/s]

Done. Embedding shape: (2949, 768)
Persisted 2949 chunks to 'data/vector_store' in collection 'drug_leaflets'.


### 2.4 Retrieval & Prompting

Retrieval embeds the question with the same model (and matching `"query: "` prefix) used to index the corpus, then does a semantic search against the vector store.

**Failure found during testing, and fix:** an initial version without drug-name filtering occasionally retrieved chunks from the *wrong* drug when several drugs share near-identical boilerplate language (e.g. NSAIDs like ibuprofen, naproxen, diclofenac, and aspirin all have very similar allergy-warning text). Embedding similarity alone can't always disambiguate which specific drug a question is about. **Fix:** since the corpus only contains 30 known drugs, the retrieval function checks whether the question names one of them and, if so, hard-filters the vector search to that drug's chunks before ranking by similarity. This eliminated the cross-drug contamination entirely on retest (verified below).


In [6]:
def embed_query(question: str) -> list[float]:
    """
    Embeds a user question with the same "query: " prefix convention
    used at index time -- mismatching this prefix silently degrades
    retrieval quality.
    """
    prefixed = f"query: {question}"
    return model.encode(prefixed, convert_to_numpy=True).tolist()


KNOWN_DRUGS = sorted(set(chunk["drug"] for chunk in all_chunks))


def detect_drug_in_question(question: str) -> str | None:
    """Checks if the question names one of the known drugs (case-insensitive)."""
    q_lower = question.lower()
    for drug in KNOWN_DRUGS:
        drug_readable = drug.replace("_", " ")
        if drug_readable in q_lower or drug in q_lower:
            return drug
    return None


def retrieve(question: str, top_k: int = 5) -> list[dict]:
    """
    Retrieves the top_k most relevant chunks for a question.
    If the question names a known drug, retrieval is hard-filtered to
    that drug's chunks first -- this prevents semantically-similar but
    wrong-drug chunks from being retrieved instead.
    """
    query_embedding = embed_query(question)
    detected_drug = detect_drug_in_question(question)

    query_kwargs = {"query_embeddings": [query_embedding], "n_results": top_k}
    if detected_drug:
        query_kwargs["where"] = {"drug": detected_drug}

    results = collection.query(**query_kwargs)

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "text": results["documents"][0][i],
            "drug": results["metadatas"][0][i]["drug"],
            "section": results["metadatas"][0][i]["section"],
            "distance": results["distances"][0][i],
        })
    return retrieved


In [7]:
test_questions = [
    "What is the recommended dosage of metformin for adults?",
    "Can I take ibuprofen while pregnant?",
    "What are the side effects of sertraline?",
    "Is it safe to take amoxicillin with alcohol?",
    "What should I do if I miss a dose of levothyroxine?",
    "What are the warning signs of an allergic reaction to aspirin?",
    "Can albuterol be used by children?",
    "What drugs interact with alprazolam?",
    "How should insulin glargine be stored?",
    "What are the contraindications for atorvastatin?",
]

for q in test_questions:
    results = retrieve(q, top_k=3)
    print(f"\nQ: {q}")
    for r in results:
        print(f"  [{r['drug']} -- {r['section']}] (dist={r['distance']:.3f})")
        print(f"    {r['text'][:150]}...")



Q: What is the recommended dosage of metformin for adults?
  [metformin -- CONTRAINDICATIONS] (dist=0.403)
    litazone and metformin hydrochloride tablets may be increased to a maximum
recommended total daily dosage of three tablets per day (45 mg of pioglitaz...
  [metformin -- DOSAGE AND ADMINISTRATION] (dist=0.418)
    Obtain liver tests before initiation. If abnormal, use caution when treating with pioglitazone and
metformin hydrochloride, investigate the probable c...
  [metformin -- CONTRAINDICATIONS] (dist=0.424)
    he next dose.
2.2 Recommended Dosage and Administration
Recommended Starting Dosage Based on Current Regimen
Individualize the starting dosage of piog...

Q: Can I take ibuprofen while pregnant?
  [ibuprofen -- WARNINGS] (dist=0.581)
    ave bloody or black stools
have stomach pain that does not get better
you have symptoms of heart problems or stroke:
chest pain
trouble breathing
weak...
  [ibuprofen -- WARNINGS] (dist=0.603)
    Allergy alert: 
Ibuprofen may cau

**Result:** with drug-name filtering in place, all 10 test questions correctly retrieve chunks from the named drug (verified: the ibuprofen and aspirin questions, which failed in an earlier version without filtering, now return only chunks tagged with the correct drug).

**Prompt template:** retrieved chunks are labeled by source and injected as context, with explicit instructions to answer only from that context and to say so when the context is insufficient — this is the actual grounding mechanism, tested against hallucination in Section 2.6.


#### Environment: Ollama (local LLM)

Ollama needs to be installed and running as a background service before generation can be tested. This cell is Colab-specific setup (a normal local machine just needs Ollama installed once, no background-process workaround needed).


In [8]:
!apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 52 not upgraded.
Need to get 1,025 kB of archives.
After this operation, 3,604 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 pci.ids all 0.0~2024.03.31-1ubuntu0.1 [275 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libpci3 amd64 1:3.10.0-2build1 [36.5 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 pciutils amd64 1:3.10.0-2build1 [69.7 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 1,025 kB in 2s (649 kB/s)
Selecting previously unselected package pci.ids.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpac

In [9]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
time.sleep(5)
print("Ollama server started.")


Ollama server started.


In [10]:
!ollama pull llama3.2:3b


In [11]:
def build_prompt(question: str, retrieved_chunks: list[dict]) -> str:
    """
    Builds a grounded prompt: retrieved chunks go in as labeled context,
    the LLM is instructed to answer ONLY from that context and cite which
    source each part of its answer came from.
    """
    context_blocks = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        drug_readable = chunk["drug"].replace("_", " ").title()
        context_blocks.append(
            f"[Source {i}: {drug_readable} -- {chunk['section'].title()}]\n{chunk['text']}"
        )
    context_text = "\n\n".join(context_blocks)

    prompt = f"""You are a medical information assistant. Answer the user's question using ONLY the information in the sources below. Do not use any outside knowledge.

If the sources do not contain enough information to answer the question, say so clearly instead of guessing.

After your answer, list which source(s) you used, like this: "Source: [drug name] -- [section name]".

--- SOURCES ---
{context_text}
--- END SOURCES ---

Question: {question}

Answer:"""
    return prompt


import ollama

OLLAMA_MODEL = "llama3.2:3b"


def generate_answer(question: str, top_k: int = 5) -> dict:
    """
    Full pipeline: retrieve -> build prompt -> call local Ollama LLM ->
    return the answer along with the sources actually retrieved (used
    for displaying citations in the frontend, independent of what the
    LLM itself claims it cited).
    """
    retrieved_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved_chunks)

    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)

    return {
        "answer": response["response"],
        "sources": [{"drug": c["drug"], "section": c["section"]} for c in retrieved_chunks],
    }


### 2.5 Vision Component

Not applicable — this project uses the **Core Track** (text-only RAG), so no CV/YOLO component is included.


### 2.6 Evaluation

Run all 10 test questions through the full `generate_answer` pipeline (retrieval + prompt + local LLM generation).


In [12]:
results_log = []

for q in test_questions:
    result = generate_answer(q, top_k=5)
    results_log.append({
        "question": q,
        "answer": result["answer"],
        "sources": result["sources"],
    })
    print(f"\n{'='*80}")
    print(f"Q: {q}")
    print(f"\nA: {result['answer']}")
    print(f"\nSources used:")
    for s in result["sources"]:
        print(f"  - {s['drug']} -- {s['section']}")



Q: What is the recommended dosage of metformin for adults?

A: I cannot provide a dosage recommendation for metformin for adults based on the provided sources. If you have any other questions about metformin, I will do my best to assist you.

Sources used:
  - metformin -- CONTRAINDICATIONS
  - metformin -- DOSAGE AND ADMINISTRATION
  - metformin -- CONTRAINDICATIONS
  - metformin -- WARNINGS AND PRECAUTIONS
  - metformin -- WARNINGS

Q: Can I take ibuprofen while pregnant?

A: According to Source 1: Ibuprofen -- Warnings, if pregnant or breast-feeding, ask a health professional before use. It is especially important not to use ibuprofen at 20 weeks or later in pregnancy unless definitely directed to do so by a doctor because it may cause problems in the unborn child or complications during delivery.

Source: [Source 1: Ibuprofen -- Warnings]

Sources used:
  - ibuprofen -- WARNINGS
  - ibuprofen -- WARNINGS
  - ibuprofen -- INACTIVE INGREDIENTS
  - ibuprofen -- WARNINGS
  - ibuprofen

**Data-quality finding during evaluation:** the metformin dosage question initially returned a confusing, self-contradictory answer. Investigating the retrieved chunks showed they kept referencing "pioglitazone and metformin hydrochloride tablets" — a **combination drug** (Actoplus Met), not plain metformin. The DailyMed search API's top result for `"Metformin"` had matched a combo product instead of the standalone label. The model wasn't reasoning poorly; it was accurately summarizing the wrong source document.

**Fix:** manually inspected the candidate SPL matches, selected a plain single-ingredient metformin label, and re-collected/re-chunked/re-embedded just that one drug (no need to redo the other 29).


In [13]:
# Inspect candidate SPL matches for "Metformin" to find a plain,
# single-ingredient label instead of the mismatched combo-drug result.
resp = requests.get(
    f"{BASE}/spls.json",
    params={"drug_name": "Metformin", "pagesize": 15, "page": 1},
    timeout=15,
)
data = resp.json()
for spl in data["data"]:
    print(f"{spl['setid']}  |  {spl['title']}")


1fa2703e-fb73-44d5-aaa2-b12b1090d42c  |  PIOGLITAZONE HYDROCHLORIDE AND METFORMIN HYDROCHLORIDE TABLET, FILM COATED [AUROBINDO PHARMA LIMITED]
73197cb5-3e23-4ac0-b189-f9bc848bce66  |  METFORMIN HYDROCHLORIDE TABLET, EXTENDED RELEASE [GRANULES INDIA LTD]
cbf8677c-cafe-48bc-8844-0cc6547886bd  |  SAXAGLIPTIN AND METFORMIN TABLET, FILM COATED, EXTENDED RELEASE [MYLAN PHARMACEUTICALS INC.]
05999192-ebc6-4198-bd1e-f46abbfb4f8a  |  METFORMIN (METFORMIN ER 500 MG) TABLET, EXTENDED RELEASE [REMEDYREPACK INC.]
7cc02a26-5c22-445b-ad8f-3e7570c143d3  |  METFORMIN HYDROCHLORIDE (METFORMIN HYDROCHLORIDE) SOLUTION [CAMBER PHARMACEUTICALS, INC.]
d5228e81-9c86-4781-b984-20feaa2ae779  |  METFORMIN HYDROCHLORIDE TABLET, EXTENDED RELEASE [REMEDYREPACK INC.]
fb832474-88d9-4e29-95cd-fbc446944cc4  |  GLUMETZA (METFORMIN HYDROCHLORIDE) TABLET, FILM COATED, EXTENDED RELEASE [SANTARUS, INC.]
237d8fa2-633c-4c29-84d3-f1b413610dc3  |  PIOGLITAZONE HCL AND METFORMIN HCL TABLET [TORRENT PHARMACEUTICALS LIMITED]
64f5b

In [14]:
# Re-download the corrected, plain-metformin label
correct_setid = "4e4c4ba9-65c6-4273-8f6b-2423ab521dad"  # METFORMIN HYDROCHLORIDE TABLET [SCIEGEN PHARMACEUTICALS INC]
out_path = Path("data/raw/metformin.pdf")
download_pdf(correct_setid, out_path)
print(f"Re-downloaded metformin.pdf with setid={correct_setid}")

# Remove the old (wrong) metformin chunks/embeddings from the vector store
old_ids = collection.get(where={"drug": "metformin"})["ids"]
collection.delete(ids=old_ids)
print(f"Removed {len(old_ids)} old metformin chunks")

# Re-chunk and re-embed the corrected file
new_chunks = build_chunks_for_drug(Path("data/raw/metformin.pdf"))
new_texts = [f"passage: {c['text']}" for c in new_chunks]
new_embeddings = model.encode(new_texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

collection.add(
    ids=[c["chunk_id"] for c in new_chunks],
    embeddings=new_embeddings.tolist(),
    documents=[c["text"] for c in new_chunks],
    metadatas=[{"drug": c["drug"], "section": c["section"]} for c in new_chunks],
)
print(f"Re-added {len(new_chunks)} corrected metformin chunks")


Re-downloaded metformin.pdf with setid=4e4c4ba9-65c6-4273-8f6b-2423ab521dad
Removed 192 old metformin chunks


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Re-added 116 corrected metformin chunks


In [15]:
# Verify the fix
result = generate_answer("What is the recommended dosage of metformin for adults?")
print(result["answer"])


The recommended starting dose of Metformin Hydrochloride Tablets for adults is 500 mg orally twice a day or 850 mg once a day, given with meals. 

Source: [Source 1: Metformin -- Dosage And Administration] -- 2.1


**Result after fix:** clean, unambiguous answer — "500 mg orally twice a day or 850 mg once a day, given with meals," correctly cited to Dosage And Administration.

### Results Table

| # | Question | Retrieved right drug/section? | Grounded or hallucinated? | Verdict |
|---|---|---|---|---|
| 1 | Metformin dosage | Yes (after data fix) | Grounded, correct | Yes (after data fix) |
| 2 | Ibuprofen + pregnancy | Yes | Grounded, correct | Yes |
| 3 | Sertraline side effects | Yes | Grounded, comprehensive | Yes |
| 4 | Amoxicillin + alcohol | Yes | Correctly abstained (no info available) | Yes |
| 5 | Levothyroxine missed dose | Yes | Correctly abstained (no specific guidance) | Yes |
| 6 | Aspirin allergic reaction | Yes | Grounded, correct | Yes |
| 7 | Albuterol in children | Yes | Grounded, correct | Yes |
| 8 | Alprazolam interactions | Yes | Grounded, detailed and accurate | Yes |
| 9 | Insulin glargine storage | Yes | Grounded, correct | Yes |
| 10 | Atorvastatin contraindications | Right drug, minor section-label mismatch | Content correct, citation label imprecise | Partially |

### Evaluation Summary

**9/10 fully correct, 1/10 partially correct** (after fixing two real issues discovered during testing).

**Failure Case 1 — Cross-drug retrieval contamination.** Initial testing surfaced failures where questions naming a specific drug retrieved chunks from a different, semantically-similar drug (e.g. an ibuprofen pregnancy question returning naproxen/diclofenac chunks). Cause: several NSAIDs share near-identical warning boilerplate, so embedding similarity alone couldn't disambiguate. **Mitigation:** added drug-name detection that hard-filters retrieval to the named drug when present in the question, eliminating this failure class entirely on retest.

**Failure Case 2 — Wrong source document (data quality).** The metformin dosage question returned a confusing, self-contradictory answer. Investigation traced this to a data collection bug: the DailyMed search API's top result for "Metformin" was a pioglitazone+metformin *combination* drug label, not plain metformin — so the model was accurately summarizing the wrong document. **Mitigation:** manually verified and re-selected the correct single-ingredient metformin label, re-chunked and re-embedded it. Retest produced a clean, correctly grounded answer.

**Minor Observation — Citation section mismatch.** The atorvastatin contraindications answer was factually correct but its self-reported citation label ("Contraindications") didn't exactly match the retrieved section metadata ("Use in Specific Populations"), where overlapping contraindication language also appears. Content was accurate; noted as a known limitation rather than corrected, since it's a labeling nuance, not a grounding error.

**Overall:** the pipeline demonstrates reliable grounding, including correctly abstaining ("no information available") rather than hallucinating when sources genuinely lack an answer (amoxicillin/alcohol, levothyroxine missed-dose questions) — arguably the most important safety property for a medical RAG assistant.


### 2.7 Export

Persist the vector store's config (embedding model, prefix convention, chunk size) alongside the store itself, so the FastAPI backend loads the exact same settings used here rather than risking a silent mismatch.


In [16]:
import json

config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "query_prefix": "query: ",
    "passage_prefix": "passage: ",
    "collection_name": COLLECTION_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "total_chunks": collection.count(),
}

config_path = f"{VECTOR_STORE_DIR}/config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved retrieval config to {config_path}")
print(json.dumps(config, indent=2))


Saved retrieval config to data/vector_store/config.json
{
  "embedding_model": "BAAI/bge-base-en-v1.5",
  "query_prefix": "query: ",
  "passage_prefix": "passage: ",
  "collection_name": "drug_leaflets",
  "chunk_size": 800,
  "chunk_overlap": 100,
  "total_chunks": 2873
}


The `data/vector_store/` folder produced by this notebook (containing `chroma.sqlite3`, `config.json`, and Chroma's internal embedding storage) is what the FastAPI backend loads at startup — copy it into `backend/data/vector_store/` before running the backend.
